In [9]:
import polars as pl

df = pl.read_parquet("./data/top_pairs/daily_top_pairs_10.parquet")

In [11]:
df

date,leader,follower,l_hat,sigma_l,cost
date,str,str,f64,f64,f64
2015-01-02,"""MRK.N""","""LOW.N""",1.050847,1.850778,0.126516
2015-01-02,"""MMM.N""","""GD.N""",-1.050328,1.936544,0.099482
2015-01-02,"""BLK.N""","""CMCSA.OQ""",-1.039419,1.976129,0.140185
2015-01-02,"""ALL.N""","""SO.N""",1.168831,2.040091,0.134003
2015-01-02,"""BIIB.OQ""","""LOW.N""",1.076923,2.061504,0.178299
…,…,…,…,…,…
2015-01-15,"""PM.N""","""AIG.N""",-1.096916,2.070208,0.148276
2015-01-15,"""GE.N""","""UNP.N""",1.092275,2.081511,0.171557
2015-01-15,"""BMY.N""","""AMGN.OQ""",-1.602564,2.082732,0.197008


In [13]:
df_bis = pl.read_parquet("./data/selected/SP100/bbo/ALL.N.parquet")

In [15]:
df_bis

timestamp,mid_price_return
"datetime[μs, America/New_York]",f64
2015-01-02 09:32:00 EST,-0.001696
2015-01-02 09:33:00 EST,0.000991
2015-01-02 09:34:00 EST,0.002122
2015-01-02 09:35:00 EST,0.000565
2015-01-02 09:36:00 EST,-0.002187
…,…
2017-03-31 15:56:00 EDT,0.000307
2017-03-31 15:57:00 EDT,-0.000429
2017-03-31 15:58:00 EDT,-0.000123


In [1]:
from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List

import numpy as np
import polars as pl

DEBUG = True
DEBUG_MAX_EVENTS = 5


# 1) Parameters


@dataclass
#essayer plusieurs d et k
class TradingParams:
    d: int = 20
    k: float = 2.5
    lag_method: str = "round"   # "round" | "floor" | "ceil" | "fractional"
    min_abs_lag: float = 0.5
    enter_on_next_bar: bool = True
    cost_is_one_way: bool = True  # if False, treat 'cost' as roundtrip (we charge cost/2 per unit change)
    fill_missing_returns_with_zero: bool = True


# 2) File loading helpers
#yassine -> pairs -> explained
def load_pairs(pairs_path: str) -> pl.DataFrame:
    """
    Load daily pairs table.
    Expected columns: date, leader, follower, l_hat, sigma_l, cost
    date can be string or date; we normalize to YYYY-MM-DD string.
    """
    if pairs_path.endswith(".parquet"):
        df = pl.read_parquet(pairs_path)
    else:
        df = pl.read_csv(pairs_path)

    if df["date"].dtype != pl.Utf8:
        df = df.with_columns(pl.col("date").cast(pl.Date).cast(pl.Utf8))

    needed = {"date", "leader", "follower", "l_hat", "sigma_l", "cost"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"pairs_df missing columns: {missing}")

    return df


def load_stock_day(returns_dir: str, ticker: str, date_str: str) -> pl.DataFrame:
    """
    Load one stock parquet filtered to one date.
    Expected columns: timestamp (tz-aware), mid_price_return (float).
    """
    path = os.path.join(returns_dir, f"{ticker}.parquet")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing parquet for ticker {ticker}: {path}")

    df = (
        pl.scan_parquet(path)
        .filter(pl.col("timestamp").dt.date().cast(pl.Utf8) == date_str)
        .select(["timestamp", "mid_price_return"])
        .collect()
    )
    return df


def get_returns_cached(cache: Dict[Tuple[str, str], pl.DataFrame],
                       returns_dir: str, ticker: str, date_str: str) -> pl.DataFrame:
    key = (ticker, date_str)
    if key not in cache:
        cache[key] = load_stock_day(returns_dir, ticker, date_str)
    return cache[key]


# 3) Trading day mapping (formation -> next)
##gives backtest loops over formation days and trades on the next day.
def build_next_day_map(unique_dates: List[str]) -> Dict[str, str]:
    """
    Given sorted unique formation dates (YYYY-MM-DD), map each date to next available date.
    This assumes your pairs_df dates are consecutive trading days in your sample.
    """
    unique_dates_sorted = sorted(unique_dates)
    m = {}
    for i in range(len(unique_dates_sorted) - 1):
        m[unique_dates_sorted[i]] = unique_dates_sorted[i + 1]
    return m


# 4) Lag discretization + signal logic

def discretize_lag(l_hat: float, method: str) -> Tuple[int, float]:
    """
    Returns (base_lag_int, frac_delta)
    - For round/floor/ceil: frac_delta=0
    - For fractional: base_lag_int=floor(l_hat), frac_delta in [0,1)
    """
    if method == "round":
        return int(np.round(l_hat)), 0.0
    if method == "floor":
        return int(np.floor(l_hat)), 0.0
    if method == "ceil":
        return int(np.ceil(l_hat)), 0.0
    if method == "fractional":
        a = int(np.floor(l_hat))
        delta = float(l_hat - a)
        return a, delta
    raise ValueError(f"Unknown lag_method: {method}")


'''
def compute_bollinger(df: pl.DataFrame, d: int, k: float) -> pl.DataFrame:
    """
    df: timestamp, rL, rF
    Adds: mu, sigma, upper, lower
    Uses shift(1) to avoid lookahead (bands at t use info through t-1).
    """
    return (
        df.with_columns([
            pl.col("rL").rolling_mean(d).alias("mu_raw"),
            pl.col("rL").rolling_std(d).alias("sigma_raw"),
        ])
        .with_columns([
            pl.col("mu_raw").shift(1).alias("mu"),
            pl.col("sigma_raw").shift(1).alias("sigma"),
        ])
        .with_columns([
            (pl.col("mu") + k * pl.col("sigma")).alias("upper"),
            (pl.col("mu") - k * pl.col("sigma")).alias("lower"),
        ])
        .drop(["mu_raw", "sigma_raw"])
    )
'''
def compute_bollinger(df: pl.DataFrame, d: int, k: float) -> pl.DataFrame:
    """
    Bands at t are computed using info through t-1 (shift(1)).
    """
    df = df.with_columns([
        pl.col("rL").rolling_mean(d).shift(1).alias("mu"),
        pl.col("rL").rolling_std(d).shift(1).alias("sigma"),
    ])

    # if sigma is null/0 early in the day, bands become null/flat -> no signal
    df = df.with_columns([
        (pl.col("mu") + k * pl.col("sigma")).alias("upper"),
        (pl.col("mu") - k * pl.col("sigma")).alias("lower"),
    ])

    return df

'''
Leads to 0 trades
def build_signal_raw(df: pl.DataFrame, cost: float) -> np.ndarray:
    """
    Raw signal from leader at time t:
      +1 if rL(t) > cost AND rL(t) > upper(t)
      -1 if rL(t) < -cost AND rL(t) < lower(t)
       0 otherwise
    Returns numpy array of ints.
    """
    sig = df.select(
        pl.when((pl.col("rL") > cost) & (pl.col("rL") > pl.col("upper"))).then(1)
        .when((pl.col("rL") < -cost) & (pl.col("rL") < pl.col("lower"))).then(-1)
        .otherwise(0)
        .alias("signal_raw")
    )["signal_raw"].to_numpy()
    return sig
'''

def build_signal_raw(df: pl.DataFrame) -> np.ndarray:
    """
    Raw signal from leader at time t:
      +1 if rL(t) > upper(t)
      -1 if rL(t) < lower(t)
       0 otherwise
    """
    sig = df.select(
        pl.when(pl.col("rL") > pl.col("upper")).then(1)
        .when(pl.col("rL") < pl.col("lower")).then(-1)
        .otherwise(0)
        .alias("signal_raw")
    )["signal_raw"].to_numpy()
    return sig



def shift_forward(x: np.ndarray, lag: int) -> np.ndarray:
    """
    Shift x forward by lag: value at t appears at t+lag.
    Creates NaN in the first lag slots.
    """
    n = len(x)
    y = np.empty(n, dtype=float)
    if lag <= 0:
        # for safety; negative lag means leader lags (we skip those pairs by default)
        return x.astype(float)
    y[:lag] = np.nan
    y[lag:] = x[:-lag]
    return y


def apply_lag(signal_raw: np.ndarray, l_hat: float, params: TradingParams) -> np.ndarray:
    """
    Apply lag to signal_raw.
    Non-fractional: returns shifted signal in {-1,0,1} (as float with NaNs at start)
    Fractional: weighted combination of lag a and a+1, output in [-1,1] with NaNs at start.
    """
    a, delta = discretize_lag(l_hat, params.lag_method)

    if params.lag_method != "fractional":
        return shift_forward(signal_raw, a)

    # fractional
    y_a = shift_forward(signal_raw, a)
    y_a1 = shift_forward(signal_raw, a + 1)
    return (1.0 - delta) * y_a + delta * y_a1


def make_minute_grid_from_df(df: pl.DataFrame) -> pl.DataFrame:
    """
    Build a master timestamp grid from an existing intraday DF.
    Assumes df already contains the full minute grid for that stock/day.
    """
    return df.select("timestamp").unique().sort("timestamp")


def align_pair_on_grid(
    grid: pl.DataFrame,
    dfL: pl.DataFrame,
    dfF: pl.DataFrame,
    fill_zero: bool
) -> pl.DataFrame:
    """
    Left-join leader/follower onto the same timestamp grid.
    Ensures every pair has identical timestamps and identical length.
    """
    out = (
        grid
        .join(dfL, on="timestamp", how="left")
        .join(dfF, on="timestamp", how="left")
    )

    if fill_zero:
        out = out.with_columns([
            pl.col("rL").fill_null(0.0),
            pl.col("rF").fill_null(0.0),
        ])

    return out


# 5) Position simulation & PnL

'''
def simulate_positions(signal_trade: np.ndarray, enter_on_next_bar: bool) -> np.ndarray:
    """
    State machine:
      - flat -> enter on non-zero signal
      - in position -> exit on 0, flip on opposite
    For fractional signals, use sign(signal).
    If enter_on_next_bar: position is delayed by 1 bar to avoid execution/lookahead.
    """
    n = len(signal_trade)
    pos = np.zeros(n, dtype=float)

    current = 0.0
    for t in range(n):
        s = signal_trade[t]
        if np.isnan(s):
            s = 0.0

        desired = 0.0
        if s > 0:
            desired = 1.0
        elif s < 0:
            desired = -1.0

        if current == 0.0:
            if desired != 0.0:
                current = desired
        else:
            if desired == 0.0:
                current = 0.0
            elif desired != current:
                current = desired

        pos[t] = current

    if enter_on_next_bar:
        pos = np.roll(pos, 1)
        pos[0] = 0.0

    return pos
'''

def simulate_positions(signal_trade: np.ndarray, enter_on_next_bar: bool) -> np.ndarray:
    """
    - flat -> enter on non-zero signal
    - in position -> exit on 0, flip on opposite
    - force flat at end of day
    """
    n = len(signal_trade)
    pos = np.zeros(n, dtype=float)

    current = 0.0
    for t in range(n):
        s = signal_trade[t]
        if np.isnan(s):
            s = 0.0

        desired = 0.0
        if s > 0:
            desired = 1.0
        elif s < 0:
            desired = -1.0

        if current == 0.0:
            if desired != 0.0:
                current = desired
        else:
            if desired == 0.0:
                current = 0.0
            elif desired != current:
                current = desired

        pos[t] = current

    if enter_on_next_bar:
        pos = np.roll(pos, 1)
        pos[0] = 0.0

    # FORCE FLAT AT EOD so exit cost is charged
    pos[-1] = 0.0

    return pos


def compute_pnl(rF: np.ndarray, pos: np.ndarray, cost: float, cost_is_one_way: bool) -> Tuple[np.ndarray, np.ndarray]:
    """
    gross = pos * rF  (pos already shifted if enter_on_next_bar)
    change = |pos[t] - pos[t-1]|
    costs:
      - one_way: cost per unit change
      - roundtrip: cost/2 per unit change
    """
    gross = pos * rF

    prev = np.roll(pos, 1)
    prev[0] = 0.0
    change = np.abs(pos - prev)

    if cost_is_one_way:
        costs = change * cost
    else:
        costs = change * (cost / 2.0)

    net = gross - costs
    return net, change


# 6) Backtest one trading day

def backtest_trade_day(
    pairs_for_formation_day: pl.DataFrame,
    trade_date: str,
    returns_dir: str,
    params: TradingParams,
    cache: Optional[Dict[Tuple[str, str], pl.DataFrame]] = None
) -> Tuple[pl.DataFrame, pl.DataFrame]:
    """
    For a given trade_date (YYYY-MM-DD), use a set of pairs (leader,follower,l_hat,cost).
    Loads data on demand (with optional cache), joins on timestamp, computes signals, positions, pnl.
    Returns:
      - minute_df: timestamp, portfolio_return
      - diag_df: pair diagnostics (num_trades, day_return, etc.)
    """
    if cache is None:
        cache = {}

    '''
    pair_pnls = []
    diag_rows = []

    for row in pairs_for_formation_day.iter_rows(named=True):
        leader = row["leader"]
        follower = row["follower"]
        l_hat = float(row["l_hat"])
        cost = float(row["cost"])

        # Skip too small or negative lags (simple version)
        if abs(l_hat) < params.min_abs_lag or l_hat <= 0:
            continue

        lag_int, lag_frac = discretize_lag(l_hat, params.lag_method)
        if params.lag_method != "fractional" and lag_int == 0:
            continue

        # load on demand
        #So for one pair:
        #load leader returns for the trade day
        #load follower returns for the trade day
        #join on timestamp
        dfL = get_returns_cached(cache, returns_dir, leader, trade_date).rename({"mid_price_return": "rL"})
        dfF = get_returns_cached(cache, returns_dir, follower, trade_date).rename({"mid_price_return": "rF"})

        df = dfL.join(dfF, on="timestamp", how="inner")

        # missing handling (if some timestamps missing after join, it's safer to keep only inner for now)
        if df.height < 50:
            # too few points -> skip
            continue

        # Bollinger on leader
        df = compute_bollinger(df, params.d, params.k)

        # raw signal and lagged trade signal
        sig_raw = build_signal_raw(df, cost)
        sig_trade = apply_lag(sig_raw, l_hat, params)

        # positions and pnl
        pos = simulate_positions(sig_trade, params.enter_on_next_bar)
        rF = df["rF"].to_numpy()
        pnl, change = compute_pnl(rF, pos, cost, params.cost_is_one_way)

        pair_pnls.append(pnl)

        diag_rows.append({
            "trade_date": trade_date,
            "leader": leader,
            "follower": follower,
            "l_hat": l_hat,
            "lag_method": params.lag_method,
            "lag_used_int": lag_int,
            "lag_frac": lag_frac,
            "cost": cost,
            "num_trades": int(np.sum(change > 0)),
            "day_return": float(np.nansum(pnl)),
            "n_points": int(df.height),
        })
    '''
    pair_pnls = []
    diag_rows = []
    minute_grid: Optional[pl.DataFrame] = None
    n_grid: Optional[int] = None

    for row in pairs_for_formation_day.iter_rows(named=True):
        leader = row["leader"]
        follower = row["follower"]
        l_hat = float(row["l_hat"])
        ocp_cost = float(row["ocp_cost"])

        # Skip too small or negative lags (your current design choice)
        if abs(l_hat) < params.min_abs_lag or l_hat <= 0:
            continue

        lag_int, lag_frac = discretize_lag(l_hat, params.lag_method)
        if params.lag_method != "fractional" and lag_int == 0:
            continue

        # load on demand
        dfL = get_returns_cached(cache, returns_dir, leader, trade_date).rename({"mid_price_return": "rL"})
        dfF = get_returns_cached(cache, returns_dir, follower, trade_date).rename({"mid_price_return": "rF"})

        # Build a MASTER minute grid once per trade day
        if minute_grid is None:
            # Use leader's timestamps as the day grid (assumes leader has full grid)
            minute_grid = make_minute_grid_from_df(dfL)
            n_grid = minute_grid.height

        # Align both series onto the same grid (NO inner join!)
        df = align_pair_on_grid(
            grid=minute_grid,
            dfL=dfL,
            dfF=dfF,
            fill_zero=params.fill_missing_returns_with_zero,
        )

        # Ensure identical length across pairs always
        if n_grid is not None and df.height != n_grid:
            # should not happen with grid joins, but keep safe
            continue

        # If you did not fill missing with zero, drop nulls consistently (optional)
        if not params.fill_missing_returns_with_zero:
            df = df.drop_nulls(["rL", "rF"])

        if df.height < max(50, params.d + 2):
            continue

        # Bollinger on leader
        df = compute_bollinger(df, params.d, params.k)

        # raw signal and lagged trade signal
        #sig_raw = build_signal_raw(df, cost)
        sig_raw = build_signal_raw(df)
        sig_trade = apply_lag(sig_raw, l_hat, params)

        # positions and pnl
        pos = simulate_positions(sig_trade, params.enter_on_next_bar)
        rF = df["rF"].to_numpy()
        tx_cost = 1e-4  # 1 bp one-way per unit position change (tune)
        pnl, change = compute_pnl(rF, pos, tx_cost, params.cost_is_one_way)

        if DEBUG and len(diag_rows) < DEBUG_MAX_EVENTS:
            print("\n================ DEBUG TRADE =================")
            print(f"Date        : {trade_date}")
            print(f"Leader      : {leader}")
            print(f"Follower    : {follower}")
            print(f"l_hat       : {l_hat:.3f}")
            print(f"lag_used    : {lag_int}")
            print("---------------------------------------------")
        
            debug_df = df.with_columns([
                pl.Series("signal_raw", sig_raw),
                pl.Series("signal_trade", sig_trade),
                pl.Series("position", pos),
                pl.Series("pnl", pnl),
            ])
        
            # Only show rows where something happens
            debug_df = debug_df.filter(
                (pl.col("signal_raw") != 0) |
                (pl.col("position").diff().abs() > 0)
            )
        
            print(
                debug_df.select([
                    "timestamp",
                    "rL",
                    "mu",
                    "upper",
                    "lower",
                    "signal_raw",
                    "signal_trade",
                    "position",
                    "rF",
                    "pnl",
                ]).head(10)
            )
        
            print("=============================================")


        # Guarantee same length for stacking
        if n_grid is not None and len(pnl) != n_grid:
            continue

        pair_pnls.append(pnl)

        diag_rows.append({
            "trade_date": trade_date,
            "leader": leader,
            "follower": follower,
            "l_hat": l_hat,
            "lag_method": params.lag_method,
            "lag_used_int": lag_int,
            "lag_frac": lag_frac,
            "ocp_cost": ocp_cost,
            "tx_cost": tx_cost,
            "num_trades": int(np.sum(change > 0)),
            "day_return": float(np.nansum(pnl)),
            "n_points": int(df.height),
        })


    if not pair_pnls:
        return pl.DataFrame(), pl.DataFrame(diag_rows)

    pnl_mat = np.vstack(pair_pnls)

    # Equal-weight portfolio, fixed weights:
    # treat NaNs (from lag shift) as 0 pnl so weights don't change over time
    pnl_mat = np.nan_to_num(pnl_mat, nan=0.0)
    port = pnl_mat.mean(axis=0)

    # Use the master grid timestamps
    minute_df = minute_grid.with_columns(pl.Series("portfolio_return", port))
    diag_df = pl.DataFrame(diag_rows)
    return minute_df, diag_df



# 7) Full run: formation day -> next day trade


def run_backtest(pairs_df: pl.DataFrame, returns_dir: str, params: TradingParams):
    """
    Treat pairs_df.date as formation day; trade on the next available date in pairs_df.
    Returns:
      - all minute portfolio returns (date, timestamp, portfolio_return)
      - all pair diagnostics (trade_date, leader, follower, ...)
      - daily portfolio returns
    """
    unique_dates = sorted(pairs_df["date"].unique().to_list())
    next_map = build_next_day_map(unique_dates)

    cache: Dict[Tuple[str, str], pl.DataFrame] = {}  # per-run cache; you can also flush per day if you want

    all_minutes = []
    all_diag = []

    for formation_day, trade_day in next_map.items():
        pairs_for_day = pairs_df.filter(pl.col("date") == formation_day)
        '''
        pairs_for_day = (
            pairs_df
            .filter(pl.col("date") == formation_day)
            .head(1)
        )
        '''

        minute_df, diag_df = backtest_trade_day(
            pairs_for_formation_day=pairs_for_day,
            trade_date=trade_day,
            returns_dir=returns_dir,
            params=params,
            cache=cache,
        )

        if minute_df.height > 0:
            minute_df = minute_df.with_columns(pl.lit(trade_day).alias("date"))
            all_minutes.append(minute_df)

        if diag_df.height > 0:
            all_diag.append(diag_df)

    minutes_out = pl.concat(all_minutes, how="vertical") if all_minutes else pl.DataFrame()
    diag_out = pl.concat(all_diag, how="vertical") if all_diag else pl.DataFrame()

    if minutes_out.height > 0:
        daily_out = (
            minutes_out
            .group_by("date")
            .agg(((pl.col("portfolio_return") + 1.0).product() - 1.0).alias("daily_return"))
            .sort("date")
        )
    else:
        daily_out = pl.DataFrame()

    return minutes_out, diag_out, daily_out


# 8) Quick diagnostic utilities


def inspect_one_stock(returns_dir: str, ticker: str, date_str: str):
    df = load_stock_day(returns_dir, ticker, date_str)
    out = df.select(
        pl.col("timestamp").min().alias("min_ts"),
        pl.col("timestamp").max().alias("max_ts"),
        pl.len().alias("n_rows")
    )
    return df, out


def no_trade_sanity(pairs_df: pl.DataFrame, returns_dir: str):
    params = TradingParams(k=999.0)  # huge bands -> no trades
    minutes, diag, daily = run_backtest(pairs_df, returns_dir, params)
    return minutes, diag, daily




PAIRS_PATH = "./data/top_pairs/daily_top_pairs_573_90.parquet"          
RETURNS_DIR = "./data/selected/SP100/bbo"     

pairs_df = load_pairs(PAIRS_PATH)
pairs_df = pairs_df.rename({"cost": "ocp_cost"})

example_date = pairs_df["date"][0]
example_ticker = pairs_df["leader"][0]

df_stock, stock_info = inspect_one_stock(RETURNS_DIR, example_ticker, example_date)
print(stock_info)

# ---- Run backtest (baseline) ----
params = TradingParams(
    d=20,
    k=2.0,
    lag_method="round",   
    min_abs_lag=0.5,
    enter_on_next_bar=True,
    cost_is_one_way=True,
)

minutes, diag, daily = run_backtest(pairs_df, RETURNS_DIR, params)

print("Daily portfolio returns:")
print(daily)

print("\nTop 10 pairs by day_return:")
if diag.height > 0:
    print(diag.sort("day_return", descending=True).head(10))

# ---- Sanity: no-trade test ----
minutes_nt, diag_nt, daily_nt = no_trade_sanity(pairs_df, RETURNS_DIR)
print("\nNo-trade sanity daily returns (should be ~0):")
print(daily_nt)


shape: (1, 3)
┌────────────────────────────────┬────────────────────────────────┬────────┐
│ min_ts                         ┆ max_ts                         ┆ n_rows │
│ ---                            ┆ ---                            ┆ ---    │
│ datetime[μs, America/New_York] ┆ datetime[μs, America/New_York] ┆ u32    │
╞════════════════════════════════╪════════════════════════════════╪════════╡
│ 2015-01-02 09:32:00 EST        ┆ 2015-01-02 16:00:00 EST        ┆ 389    │
└────────────────────────────────┴────────────────────────────────┴────────┘

================ DEBUG TRADE =================
Date        : 2015-01-05
Leader      : MRK.N
Follower    : LOW.N
l_hat       : 1.051
lag_used    : 1
---------------------------------------------
shape: (10, 10)
┌─────────────────────────┬───────────┬───────────┬──────────┬───┬──────────────┬──────────┬───────────┬───────────┐
│ timestamp               ┆ rL        ┆ mu        ┆ upper    ┆ … ┆ signal_trade ┆ position ┆ rF        ┆ pnl       │
│ 